In [1]:
"""
GraphSAGE on a kNN graph for rare-event node classification
-----------------------------------------------------------
- Builds a kNN graph from your feature matrix X (R^{N x d})
- Trains a GraphSAGE GNN with BCEWithLogitsLoss and pos_weight for imbalance
- Reports ROC-AUC and PR-AUC (Average Precision) and saves diagnostic plots

Usage
-----
1) Set `use_files = True` in `main()` to read your files with `_load_data_to_mem`.
2) Fill `parameter_config` and `files` accordingly.
3) Run:  python knn-graphsage-node-classification.py

Dependencies
------------
- torch, torch_geometric (PyG)
- numpy, pandas, h5py
- scikit-learn
- matplotlib (for plots)
"""
from __future__ import annotations
import os
import random
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, Tuple, Sequence, Union
from resolve.utilities import utilities as utils
import yaml
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler
import h5py
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    roc_curve
)

from torch_geometric.data import Data
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import add_self_loops, to_undirected

import matplotlib.pyplot as plt

In [2]:
import hnswlib, numpy as np, torch
from torch_geometric.utils import to_undirected, add_self_loops

def hnsw_knn_edges(X_np: np.ndarray, k: int = 5, space='cosine', ef_construction=200, M=32, ef=200):
    N, d = X_np.shape
    p = hnswlib.Index(space=space, dim=d)
    p.init_index(max_elements=N, ef_construction=ef_construction, M=M)
    p.add_items(X_np, np.arange(N))
    p.set_ef(ef)                 # query-time accuracy/speed tradeoff
    I, _ = p.knn_query(X_np, k=k+1)  # includes self
    I = I[:, 1:]                 # drop self
    src = np.repeat(np.arange(N), k)
    dst = I.reshape(-1)
    edge_index = torch.tensor(np.vstack([src, dst]), dtype=torch.long)
    edge_index = to_undirected(edge_index, num_nodes=N)
    edge_index, _ = add_self_loops(edge_index, num_nodes=N)
    return edge_index

In [3]:
def init_head_bias_from_prior(model, mask_train, data):
        with torch.no_grad():
            p = data.y[mask_train].float().mean().clamp(1e-6, 1-1e-6).item()
            b = np.log(p/(1-p))
            model.head.bias.fill_(-b)

In [4]:
def focal_bce_with_logits(logits, targets, alpha=0.5, gamma=2.0):
    p = torch.sigmoid(logits).clamp(1e-6, 1-1e-6)
    loss_pos = -alpha * ((1-p)**gamma) * targets * torch.log(p)
    loss_neg = -(1-alpha) * (p**gamma) * (1-targets) * torch.log(1-p)
    return (loss_pos + loss_neg).mean()

In [5]:
from torch_geometric.utils import dropout_edge
def dropedge(edge_index, p=0.2, training=True):
    if training:
        ei, _ = dropout_edge(edge_index, p=p, force_undirected=False)
        return ei
    return edge_index

In [6]:



# ------------------------------
# Repro
# ------------------------------
from torch import logit


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def _get_hdf5_files(path_to_files, config_file):
        return sorted(str(p) for p in path_to_files.glob(f"*.{config_file['simulation_settings']['file_format']}"))

# ------------------------------
# Data utilities + file pipeline
# ------------------------------
def _read_in_from_file(file_path: str, parameter_config: Dict) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Read (theta, phi, y) from HDF5 or CSV and return tensors (x, y).
    - x = [theta | phi] concatenated along last dimension, float32
    - y shaped (N, 1) float32

    parameter_config can provide either 'key'+'selected_indices' (for HDF5) or 'selected_labels' (for CSV):
      'phi':   {'key' or 'selected_labels', 'selected_indices'}
      'theta': {'key' or 'selected_labels', 'selected_indices'}
      'target':{'key' or 'selected_labels', 'selected_indices'}
    """
    if file_path.endswith(('.h5', '.hdf5')):
        with h5py.File(file_path, 'r') as hdf:
            # φ
            phi = hdf[parameter_config['phi']['key']][:, parameter_config['phi']['selected_indices']]
            # θ
            theta = hdf[parameter_config['theta']['key']]
            if len(parameter_config['theta']['selected_indices']) != 0:
                if theta.ndim == 1:
                    theta_vec = theta[parameter_config['theta']['selected_indices']]  # (T,)
                    theta = torch.from_numpy(theta_vec).unsqueeze(0).expand(phi.shape[0], -1)
                else:
                    theta = theta[:, parameter_config['theta']['selected_indices']]
                    theta = torch.from_numpy(theta)
            else:
                theta = torch.from_numpy(theta)

            # y / target
            tgt_ds = hdf[parameter_config['target']['key']]
            if tgt_ds.ndim > 1 and parameter_config['target']['selected_indices'] is not None:
                y = tgt_ds[:, parameter_config['target']['selected_indices']]
            else:
                y = tgt_ds[:].reshape(-1, 1)

        phi = torch.from_numpy(phi)
        y = torch.from_numpy(y)
        x = torch.cat([theta, phi], dim=-1)

    elif file_path.endswith('.csv'):
        # CSV via selected_labels
        df = pd.read_csv(file_path)

        def select_labels(df_: pd.DataFrame, labels: Union[str, Sequence[str]]) -> pd.DataFrame:
            if isinstance(labels, str):
                return df_[[labels]]
            elif isinstance(labels, (list, tuple)):
                return df_[list(labels)]
            else:
                raise ValueError(f"Invalid label type: {type(labels)}")

        phi_df = select_labels(df, parameter_config['phi']['selected_labels'])
        theta_df = select_labels(df, parameter_config['theta']['selected_labels'])
        y_df = select_labels(df, parameter_config['target']['selected_labels'])

        phi = torch.tensor(phi_df.values, dtype=torch.float32)
        theta = torch.tensor(theta_df.values, dtype=torch.float32)
        y = torch.tensor(y_df.values, dtype=torch.float32)

        if y.ndim == 1:
            y = y.unsqueeze(1)

        x = torch.cat([theta, phi], dim=-1)

    else:
        raise ValueError(f"Unsupported file format: {file_path}")

    # ensure float32 tensors
    x = x.contiguous().to(torch.float32)
    y = y.contiguous().to(torch.float32)
    return x, y


def _load_data_to_mem(files: Sequence[str], cfg: Dict) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    X, ys, file_inds = [], [], []
    for i, fp in enumerate(files):
        if not os.path.exists(fp):
            raise FileNotFoundError(fp)
        Xi, yi = _read_in_from_file(fp, cfg)
        X.append(Xi)
        ys.append(yi)
        file_inds.append(torch.full((Xi.size(0),), i, dtype=torch.long))
    x = torch.cat(X, 0).contiguous()
    y = torch.cat(ys, 0).contiguous()
    fidx = torch.cat(file_inds, 0).contiguous()
    return x, y, fidx


# ------------------------------
# kNN Graph
# ------------------------------
def build_knn_graph(X: np.ndarray, k: int = 10, metric: str = "euclidean") -> torch.Tensor:
    """Return edge_index (2, E) for an undirected kNN graph."""
    assert X.ndim == 2, "X must be of shape [N, d]"
    N = X.shape[0]
    nbrs = NearestNeighbors(n_neighbors=min(k + 1, N), algorithm="auto", metric=metric)
    nbrs.fit(X)
    distances, indices = nbrs.kneighbors(X)

    # indices[:, 0] is the point itself (distance=0). Drop it.
    ind = indices[:, 1:]

    # Build directed edges i -> j for each neighbor j in ind[i]
    src = np.repeat(np.arange(N), ind.shape[1])
    dst = ind.reshape(-1)
    edge_index = np.stack([src, dst], axis=0)

    # Make undirected and unique
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_index = to_undirected(edge_index)

    # Add self-loops for stability
    edge_index, _ = add_self_loops(edge_index, num_nodes=N)
    return edge_index


def make_pyg_data(X: np.ndarray, y: np.ndarray, edge_index: torch.Tensor) -> Data:
    X_t = torch.tensor(X, dtype=torch.float32)
    # Store labels as long (0/1); convert to float when computing BCE
    y_t = torch.tensor(y.astype(np.int64).ravel(), dtype=torch.long).view(-1)
    data = Data(x=X_t, edge_index=edge_index, y=y_t)
    return data


# ------------------------------
# Model
# ------------------------------
class GraphSAGE(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dims: list[int] = [128, 128],
        dropout: float = 0.2,
        debug_std: bool = False,        # <— turn on to print layer stds
        residual: bool = True           # <— enable residual skips to fight oversmoothing
    ):
        super().__init__()
        dims = [in_dim] + hidden_dims
        self.convs = nn.ModuleList([SAGEConv(dims[i], dims[i + 1]) for i in range(len(dims) - 1)])
        self.norms = nn.ModuleList([nn.LayerNorm(d) for d in hidden_dims])
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_dims[-1], 1)

        # For residuals when dimensions change, use linear skips
        self.residual = residual
        self.skips = nn.ModuleList([
            (nn.Identity() if dims[i] == dims[i+1] else nn.Linear(dims[i], dims[i+1]))
            for i in range(len(dims) - 1)
        ])

        self.debug_std = debug_std
    

    def forward(self, x, edge_index):
        h = x
        for layer_idx, (conv, norm, skip) in enumerate(zip(self.convs, self.norms, self.skips)):
            h_in = h

            # ---- message passing ----
            h = conv(h, edge_index)

            # ---- (optional) diagnostics: print variance right AFTER conv ----
            if self.debug_std:
                # std before nonlinearity gives you a real oversmoothing signal
                with torch.no_grad():
                    s = h.std().item()
                print(f"[GNN] layer {layer_idx} pre-activation std: {s:.4f}")

            # ---- residual skip to combat oversmoothing ----
            if self.residual:
                h = h + skip(h_in)

            # ---- norm + nonlinearity + dropout ----
            h = norm(h)
            h = F.relu(h)
            h = self.dropout(h)

        logits = self.head(h).squeeze(-1)
        return logits


# ------------------------------
# Training / Eval
# ------------------------------
@dataclass
class TrainConfig:
    lr: float = 3e-4
    weight_decay: float = 1e-4
    epochs: int = 150
    pos_weight_cap: float = 100.0  # cap extreme imbalance
    print_every: int = 10


def _ensure_dir(path: str):
    os.makedirs(path, exist_ok=True)


def plot_pr_curve(y_true_np, probs_np, outdir="plots", title_prefix="Test"):
    _ensure_dir(outdir)
    precision, recall, _ = precision_recall_curve(y_true_np, probs_np)
    plt.figure()
    plt.plot(recall, precision)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"{title_prefix} PR Curve")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{title_prefix.lower().replace(' ', '_')}_pr_curve.png"), dpi=150)
    plt.close()


def plot_roc_curve(y_true_np, probs_np, outdir="plots", title_prefix="Test"):
    _ensure_dir(outdir)
    fpr, tpr, _ = roc_curve(y_true_np, probs_np)
    plt.figure()
    plt.plot(fpr, tpr)
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.title(f"{title_prefix} ROC Curve")
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{title_prefix.lower().replace(' ', '_')}_roc_curve.png"), dpi=150)
    plt.close()


def plot_score_hist(probs_np, y_true_np, outdir="plots", title_prefix="Test"):
    _ensure_dir(outdir)
    plt.figure()
    plt.hist(probs_np[y_true_np == 0], bins=50, alpha=0.7, label="neg")
    plt.hist(probs_np[y_true_np == 1], bins=50, alpha=0.7, label="pos")
    plt.xlabel("Predicted probability")
    plt.ylabel("Count")
    plt.yscale('log')
    plt.title(f"{title_prefix} Score Histogram")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(outdir, f"{title_prefix.lower().replace(' ', '_')}_score_hist.png"), dpi=150)
    plt.close()


def train(model: nn.Module, data: Data, mask_train: torch.Tensor, mask_val: torch.Tensor,
          cfg: TrainConfig, device: str | None = None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    data = data.to(device)

    # Compute pos_weight for BCE from train mask
    y_train = data.y[mask_train].float()
    n_pos = y_train.sum().item()
    n_neg = y_train.numel() - n_pos
    if n_pos == 0:
        raise ValueError("No positive labels in the training split.")
    pos_weight = torch.tensor([min(cfg.pos_weight_cap, n_neg / max(1.0, n_pos))], device=device)

    #criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    criterion = lambda z,t: focal_bce_with_logits(z,t,alpha=0.99,gamma=2.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    def _metrics(split_mask: torch.Tensor):
        model.eval()
        with torch.no_grad():
            logits = model(data.x, data.edge_index)
            logits = -logits
            logits = logits[split_mask]
            y_true = data.y[split_mask].float()
            probs = torch.sigmoid(logits).detach().cpu().numpy()
            y_np = y_true.detach().cpu().numpy()
            try:
                roc = roc_auc_score(y_np, probs)
            except ValueError:
                roc = float("nan")
            try:
                pr = average_precision_score(y_np, probs)
            except ValueError:
                pr = float("nan")
            loss_val = criterion(logits, y_true).item()

            roc  = roc_auc_score(y_np, probs);     pr  = average_precision_score(y_np, probs)
            rocF = roc_auc_score(y_np, 1-probs);   prF = average_precision_score(y_np, 1-probs)
            #print(f"ROC {roc:.3f} PR {pr:.4f} | FLIPPED ROC {rocF:.3f} PR {prF:.4f}")
        return {"loss": loss_val, "roc_auc": roc, "pr_auc": pr}

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        #logits = model(data.x, data.edge_index)
        edge_idx = dropedge(data.edge_index, p=0.2, training=model.training)
        logits = model(data.x, edge_idx)
        loss = criterion((-logits)[mask_train], data.y[mask_train].float())
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        optimizer.step()

        if epoch % cfg.print_every == 0 or epoch == 1 or epoch == cfg.epochs:
            train_metrics = _metrics(mask_train)
            val_metrics = _metrics(mask_val)
            print(
                f"Epoch {epoch:03d} | Train loss {train_metrics['loss']:.4f} | "
                f"Val loss {val_metrics['loss']:.4f} | "
                f"Val ROC-AUC {val_metrics['roc_auc']:.4f} | Val PR-AUC {val_metrics['pr_auc']:.4f}"
            )

    return model


# ------------------------------
# Example: synthetic data (replace with your own)
# ------------------------------
def make_toy_data(N=4000, d=5, pos_frac=0.01, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(N, d))
    # Thin-manifold positive class: a curved filament near a nonlinear curve
    t = rng.normal(0, 1, size=N)
    curve = np.stack([np.sin(t), np.cos(t), 0.5 * t, 0.1 * t ** 2, np.sin(2 * t)], axis=1)
    dist = np.linalg.norm(X - curve, axis=1)
    y = (dist < np.quantile(dist, pos_frac)).astype(np.int64)
    return X, y


# ------------------------------
# Main
# ------------------------------
def main():
    set_seed(123)

    # 1) Load or create your data
    # === Option A: from files (HDF5/CSV). Set use_files=True and configure ===
    use_files = True  # <- set True to use your files
    if use_files:
        # Example parameter_config (edit to your keys/labels/indices)
        path_to_settings = "./binary-black-hole"
        with open(f"{path_to_settings}/settings.yaml", "r") as f:
            config_file = yaml.safe_load(f)
        sim = config_file["simulation_settings"]
        parameters = {
            "phi":    {"key": "phi",    "label_key": "phi_labels",    "selected_labels": sim["phi_labels"],    "size": len(sim["phi_labels"]),      "selected_indices": None},
            "theta":  {"key": "theta",  "label_key": "theta_headers", "selected_labels": sim["theta_labels"],  "size": len(sim["theta_labels"]),  "selected_indices": None},
            "target": {"key": "target", "label_key": "target_headers","selected_labels": sim["target_labels"], "size": len(sim["target_labels"]), "selected_indices": None},
        }
        files = _get_hdf5_files(Path(config_file["path_settings"]["path_to_files_train"]), config_file)

        if files[0].endswith(('.h5', '.hdf5')):
            parameters["phi"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["phi"])
            parameters["target"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["target"])
            parameters["theta"]["selected_indices"] = utils.find_selected_indices(files[0],parameters["theta"])


        X_t, y_t, _ = _load_data_to_mem(files, parameters)
        # Ensure numpy arrays for kNN builder

        # X is your raw numpy or torch input, shape [N, D]
        scaler = StandardScaler()

        # Convert to numpy if needed
        X_np = X_t.cpu().numpy() if isinstance(X_t, torch.Tensor) else X_t
        #X_t = X.detach().clone().to(torch.float32) if isinstance(X, torch.Tensor) else torch.tensor(X, dtype=torch.float32)

        # Fit on ALL data (or train split only)
        X_np = scaler.fit_transform(X_np)

        # Back to torch
        X = torch.tensor(X_np, dtype=torch.float32)
        y_arr = y_t.cpu().numpy().reshape(-1, 1)

        # Ensure binary labels {0,1}. If multi-target or continuous, map/threshold here.
        if y_arr.shape[1] > 1:
            # choose a column or reduce to a binary indicator
            y = (y_arr[:, 0] > 0.5).astype(np.int64)
        else:
            if not np.array_equal(np.unique(y_arr), np.array([0, 1])):
                y = (y_arr[:, 0] > 0.5).astype(np.int64)
            else:
                y = y_arr[:, 0].astype(np.int64)
        N, d = X.shape
    else:
        # === Option B: synthetic toy data ===
        X, y = make_toy_data(N=6000, d=5, pos_frac=0.007)
        N, d = X.shape

    # 2) Build a kNN graph in the *same* feature space you plan to learn on
    K = 3  # try 5-15 for sharp manifolds
    #print("build_knn_graph")
    #edge_index = build_knn_graph(X, k=K, metric="euclidean")
    # x is torch.Tensor [N, d] after StandardScaler
    print("Building HNSW graph...")

    edge_index = hnsw_knn_edges(
        X,
        k=3,                # keep very small to avoid oversmoothing
        ef_construction=200,
        M=32,
        ef=100              # lowers RAM and speeds querying
    )
    print("Graph built:", edge_index.size())

    # 3) Create PyG data object
    print("make_pyg_data")
    data = make_pyg_data(X, y, edge_index)

    # 4) Train/val/test split (stratified)
    idx = np.arange(N)
    idx_train, idx_tmp, y_train, y_tmp = train_test_split(
        idx, y, test_size=0.4, stratify=y, random_state=42
    )
    idx_val, idx_test, y_val, y_test = train_test_split(
        idx_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42
    )

    mask_train = torch.zeros(N, dtype=torch.bool)
    mask_val = torch.zeros(N, dtype=torch.bool)
    mask_test = torch.zeros(N, dtype=torch.bool)
    mask_train[idx_train] = True
    mask_val[idx_val] = True
    mask_test[idx_test] = True

    # 5) Model
    model = GraphSAGE(in_dim=d, hidden_dims=[128,128], dropout=0.2, debug_std=False)
    init_head_bias_from_prior(model, mask_train, data)

    # 6) Train
    cfg = TrainConfig(lr=3e-4, weight_decay=1e-4, epochs=50, pos_weight_cap=100, print_every=1)
    model = train(model, data, mask_train, mask_val, cfg)

    # 7) Final evaluation on test + plots
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    data = data.to(device)
    model.eval()
    with torch.no_grad():
        logits = -model(data.x, data.edge_index)[mask_test]
        y_true = data.y[mask_test].float()
        probs = torch.sigmoid(logits).cpu().numpy()
        y_np = y_true.cpu().numpy()
        roc = roc_auc_score(y_np, probs)
        pr = average_precision_score(y_np, probs)
        roc  = roc_auc_score(y_np, probs);     pr  = average_precision_score(y_np, probs)
        rocF = roc_auc_score(y_np, 1-probs);   prF = average_precision_score(y_np, 1-probs)

        #print(f"ROC {roc:.3f} PR {pr:.4f} | FLIPPED ROC {rocF:.3f} PR {prF:.4f}")
        print(f"Test ROC-AUC: {roc:.4f} | Test PR-AUC: {pr:.4f}")

        # Save diagnostic plots
        y_bin = y_np.astype(int)
        plot_pr_curve(y_bin, probs, outdir="plots", title_prefix="Test")
        plot_roc_curve(y_bin, probs, outdir="plots", title_prefix="Test")
        plot_score_hist(probs, y_bin, outdir="plots", title_prefix="Test")
        print("Saved plots to ./plots: test_pr_curve.png, test_roc_curve.png, test_score_hist.png")

                # after computing probs and y_np
        
        
if __name__ == "__main__":
    main()

Building HNSW graph...
Graph built: torch.Size([2, 24662789])
make_pyg_data


/var/folders/cx/hc0wcsld7534blvnxwn3wt6w0000gn/T/ipykernel_50109/3757354051.py:137: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X_t = torch.tensor(X, dtype=torch.float32)


Epoch 001 | Train loss 0.0328 | Val loss 0.0328 | Val ROC-AUC 0.6247 | Val PR-AUC 0.0130
Epoch 002 | Train loss 0.0318 | Val loss 0.0318 | Val ROC-AUC 0.6336 | Val PR-AUC 0.0133
Epoch 003 | Train loss 0.0309 | Val loss 0.0308 | Val ROC-AUC 0.6420 | Val PR-AUC 0.0137
Epoch 004 | Train loss 0.0299 | Val loss 0.0298 | Val ROC-AUC 0.6500 | Val PR-AUC 0.0141
Epoch 005 | Train loss 0.0289 | Val loss 0.0289 | Val ROC-AUC 0.6577 | Val PR-AUC 0.0144
Epoch 006 | Train loss 0.0280 | Val loss 0.0279 | Val ROC-AUC 0.6650 | Val PR-AUC 0.0147
Epoch 007 | Train loss 0.0271 | Val loss 0.0270 | Val ROC-AUC 0.6720 | Val PR-AUC 0.0151
Epoch 008 | Train loss 0.0262 | Val loss 0.0261 | Val ROC-AUC 0.6787 | Val PR-AUC 0.0155
Epoch 009 | Train loss 0.0253 | Val loss 0.0252 | Val ROC-AUC 0.6853 | Val PR-AUC 0.0159
Epoch 010 | Train loss 0.0244 | Val loss 0.0243 | Val ROC-AUC 0.6917 | Val PR-AUC 0.0164
Epoch 011 | Train loss 0.0235 | Val loss 0.0235 | Val ROC-AUC 0.6981 | Val PR-AUC 0.0168
Epoch 012 | Train los